# HairNet Compliance Detection

### Model Registry

1. Registers the **top 4 models** (selected by lowest `val/cls_loss`) into the MLflow Model Registry
2. Transitions the one with the highest **test/mAP50** to **Production**
3. Transitions the rest to **Archived**

In [ ]:
!pip install mlflow ultralytics -q

## 1. Imports & Setup

In [4]:
import mlflow
from mlflow import MlflowClient
import pandas as pd

MLFLOW_TRACKING_URI = 'mlruns'
EXPERIMENT_NAME     = 'hairnet-compliance-detection'
REGISTERED_NAME     = 'hairnet-compliance-detector'

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = MlflowClient()

RUN_NAMES = [
    'yolov26m-baseline',
    'yolo12m-baseline',
    'yolov26m-augmented',
    'yolo12m-augmented',
]

print(f'Tracking URI : {mlflow.get_tracking_uri()}')
print(f'Registry name: {REGISTERED_NAME}')

Tracking URI : mlruns
Registry name: hairnet-compliance-detector


## 2. Register All 8 Models

Each `best.pt` is registered as a new version under the shared name `hairnet-compliance-detector`.

In [5]:
from pathlib import Path

registered_versions = []

for run_name in RUN_NAMES:
    # ── Look up run by name
    runs_df = mlflow.search_runs(
        experiment_names=[EXPERIMENT_NAME],
        filter_string=f"tags.`mlflow.runName` = '{run_name}'",
    )
    if runs_df.empty:
        print(f'[SKIP] Run not found: {run_name}')
        continue

    run_id = runs_df.iloc[0]['run_id']

    # ── Resolve best.pt path 
    exp_id = mlflow.get_run(run_id).info.experiment_id
    best_pt_path = Path('mlruns') / exp_id / run_id / 'artifacts' / 'weights' / 'best.pt'
    if not best_pt_path.exists():
        print(f'[SKIP] best.pt not found: {run_name}')
        continue

    # ── Register directly — no run modification
    mv = client.create_model_version(
        name=REGISTERED_NAME,
        source=str(best_pt_path),
        run_id=run_id,
        tags={'run_name': run_name},
    )

    registered_versions.append((run_name, run_id, mv.version))
    print(f'Registered  {run_name:<25}  →  version {mv.version}')

print(f'\nTotal registered: {len(registered_versions)} versions')

Registered  yolov26m-baseline          →  version 1
Registered  yolo12m-baseline           →  version 2
Registered  yolov26m-augmented         →  version 3
Registered  yolo12m-augmented          →  version 4

Total registered: 4 versions


## 3. Pick Best Model by test/mAP50 → Promote to Production

In [6]:
rows = []
for run_name, run_id, version in registered_versions:
    runs_df = mlflow.search_runs(
        experiment_names=[EXPERIMENT_NAME],
        filter_string=f"tags.`mlflow.runName` = '{run_name}'",
    )
    test_map50 = runs_df.iloc[0].get('metrics.test/mAP50', 0.0)
    rows.append({'run_name': run_name, 'version': version, 'test/mAP50': test_map50})

df = pd.DataFrame(rows).sort_values('test/mAP50', ascending=False).reset_index(drop=True)
df.index += 1
print(df.to_string())

best_row = df.iloc[0]
best_ver = str(int(best_row['version']))
best_name = best_row['run_name']
print(f'\nBest model: {best_name}  (version {best_ver}, test/mAP50={best_row["test/mAP50"]:.4f})')

             run_name  version  test/mAP50
1   yolo12m-augmented        4    0.994308
2  yolov26m-augmented        3    0.992706
3    yolo12m-baseline        2    0.990765
4   yolov26m-baseline        1    0.987570

Best model: yolo12m-augmented  (version 4, test/mAP50=0.9943)


In [7]:
# Transition best version → Production, all others → Archived
for run_name, run_id, version in registered_versions:
    ver_str = str(int(version))
    if ver_str == best_ver:
        client.transition_model_version_stage(
            name=REGISTERED_NAME, version=ver_str, stage='Production',
            archive_existing_versions=False,
        )
        print(f'  Production  ←  version {ver_str}  ({run_name})')
    else:
        client.transition_model_version_stage(
            name=REGISTERED_NAME, version=ver_str, stage='Archived',
            archive_existing_versions=False,
        )
        print(f'  Archived    ←  version {ver_str}  ({run_name})')

print('\nModel Registry updated.')

  Archived    ←  version 1  (yolov26m-baseline)
  Archived    ←  version 2  (yolo12m-baseline)
  Archived    ←  version 3  (yolov26m-augmented)
  Production  ←  version 4  (yolo12m-augmented)

Model Registry updated.


C:\Users\wajee\AppData\Local\Temp\ipykernel_9948\684995857.py:11: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(
C:\Users\wajee\AppData\Local\Temp\ipykernel_9948\684995857.py:5: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


## 4. Verify — Load Production Model from Registry

In [8]:
# Confirm what is in Production right now
prod_versions = client.get_latest_versions(REGISTERED_NAME, stages=['Production'])
for v in prod_versions:
    print(f'Production model')
    print(f'  Name    : {v.name}')
    print(f'  Version : {v.version}')
    print(f'  Run ID  : {v.run_id}')
    print(f'  Stage   : {v.current_stage}')
    print(f'  Tags    : {v.tags}')

Production model
  Name    : hairnet-compliance-detector
  Version : 4
  Run ID  : 2226196a7be04073907bd58474fa1189
  Stage   : Production
  Tags    : {'run_name': 'yolo12m-augmented'}


C:\Users\wajee\AppData\Local\Temp\ipykernel_9948\642179127.py:2: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  prod_versions = client.get_latest_versions(REGISTERED_NAME, stages=['Production'])
